# ML-07 ? Baseline Action Score and Top-20 Review

## Objective

Create a transparent rule-based queue to help editors prioritize pages for human review.

This is a decision-support baseline, not a trained model and not proof that editing a page will improve its performance. The starter dataset is a single snapshot, so the available decline label is only a current-snapshot proxy.

## Dataset

The analysis uses `data/raw/content_refresh_anonymized.csv` (30,000 content rows). Pseudonymous content IDs are retained only to identify items in the output queue.


## 1. My rule and its reason codes

### Signal checks and verdicts

**Signal 1 ? Staleness / refresh: MIXED**

The 91?180-day update group has median impressions of 1,692, compared with 470 for pages updated in the last 30 days. However, the 181+ day group has only 15.5 median impressions. Age since update does not consistently identify valuable pages. Staleness alone is therefore not a reliable prioritization rule.

**Signal 2 ? CTR versus search position: CONFIRMED**

Among pages with at least 300 impressions and a valid average position, median CTR generally falls as position worsens: 0.23% at positions 4?10, 0.17% at 11?20, 0.08% at 21?50, and 0.00% beyond 50. The top-3 bucket is a small exception to a perfectly monotonic pattern. This supports checking pages outside the first 10 positions, while recognizing that CTR also depends on query mix and other factors.

### Rule in plain words

Prioritize pages with at least 300 impressions in the trailing 90 days and an average search position greater than 10 and no greater than 50. Rank eligible pages by their 90-day impressions, highest first. This gives editors a queue of visible pages outside the first 10 positions; it does not claim that every page is declining.

**Score:** `impressions_90d` for eligible pages; zero for pages outside the eligibility rule.

**One reason code:** `visible_striking_distance`

**Action label:** `review_title_content_and_search_snippet`

The action is a human review of the title, content relevance, and search snippet. No automatic content change is recommended.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "data/raw/content_refresh_anonymized.csv").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

DATA_PATH = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found. Checked from: {Path.cwd()}")

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Unique content IDs:", df["content_id"].nunique())

# Signal 1: days since last update, linked to refresh/staleness logic.
df["freshness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

staleness_table = (
    df.groupby("freshness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions=("impressions_90d", "median"),
          median_clicks=("clicks_90d", "median")
      )
)

print("\nSIGNAL 1 ? STALENESS / REFRESH")
print(staleness_table.to_string())

# Signal 2: CTR by search-position bucket.
# Exclude avg_position=0 (no position data) and apply a 300-impression floor.
position_data = df[
    (df["avg_position"] > 0) &
    (df["impressions_90d"] >= 300)
].copy()

position_data["position_bucket"] = pd.cut(
    position_data["avg_position"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=["Top 3", "4-10", "11-20", "21-50", "50+"]
)

ctr_table = (
    position_data.groupby("position_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_ctr=("ctr", "median"),
          mean_ctr=("ctr", "mean"),
          median_impressions=("impressions_90d", "median")
      )
)

print("\nSIGNAL 2 ? CTR VS POSITION")
print("Eligibility: avg_position > 0 and impressions_90d >= 300")
print(ctr_table.to_string())

print("\nSIGNAL CHECKS")
print("Staleness verdict: MIXED")
print("CTR vs position verdict: CONFIRMED")
print("Rows with avg_position = 0 (no position data):",
      int((df["avg_position"] == 0).sum()))


Dataset shape: (30000, 44)
Unique content IDs: 30000

SIGNAL 1 ? STALENESS / REFRESH
                      n  median_impressions  median_clicks
freshness_bucket                                          
0-30 days         20480               470.0            1.0
31-90 days          175               510.0            0.0
91-180 days        9171              1692.0            2.0
181+ days           174                15.5            0.0

SIGNAL 2 ? CTR VS POSITION
Eligibility: avg_position > 0 and impressions_90d >= 300
                    n  median_ctr  mean_ctr  median_impressions
position_bucket                                                
Top 3             507        0.20  0.339073              3685.0
4-10             7647        0.23  0.336940              3872.0
11-20            5051        0.17  0.259749              1783.0
21-50            4985        0.08  0.139972              1859.0
50+               562        0.00  0.043381               728.5

SIGNAL CHECKS
Staleness ver

## 2. Build the ranked queue

The score uses only the two eligibility conditions and `impressions_90d`. It does not use trend fields, labels, identifiers, or future-window information.

Every qualifying row receives the same reason code and action label. Ties are resolved by pseudonymous content ID for reproducibility.

The notebook writes `work/outputs/baseline_action_score.csv`. This generated CSV is intentionally not committed to Git; rerunning the notebook recreates it.


In [2]:
# Transparent rule: visible pages ranking between positions 11 and 50.
MIN_IMPRESSIONS = 300

candidates = df[
    (df["impressions_90d"] >= MIN_IMPRESSIONS) &
    (df["avg_position"] > 10) &
    (df["avg_position"] <= 50)
].copy()

candidates["score"] = candidates["impressions_90d"].astype(float)
candidates["reason_code"] = "visible_striking_distance"
candidates["action"] = "review_title_content_and_search_snippet"

candidates = candidates.sort_values(
    ["score", "content_id"],
    ascending=[False, True]
).reset_index(drop=True)

candidates["rank"] = np.arange(1, len(candidates) + 1)

output_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "action",
    "content_type",
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update"
]

output_path = REPO_ROOT / "work/outputs/baseline_action_score.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
candidates[output_columns].to_csv(output_path, index=False)

print("Rule: impressions >= 300 and 10 < avg_position <= 50")
print("Score: impressions_90d for eligible pages")
print("Reason code: visible_striking_distance")
print("Action: review_title_content_and_search_snippet")
print("\nTotal dataset rows:", len(df))
print("Qualifying candidates:", len(candidates))
print("Candidate share: {:.2f}%".format(len(candidates) / len(df) * 100))
print("CSV written:", output_path.as_posix())

# The trend-derived proxy is used ONLY for evaluation, never for scoring.
# It is not a verified future outcome because this is a single snapshot.
all_proxy_labels = (df["trend_direction"] == "down").astype(int)
candidate_proxy_labels = (
    candidates["trend_direction"] == "down"
).astype(int)

def precision_at_k(labels, k):
    return float(labels.iloc[:k].mean())

print("\nEVALUATION ? CURRENT-SNAPSHOT DECLINE PROXY ONLY")
print("Dataset proxy base rate: {:.3f}".format(all_proxy_labels.mean()))
print("Candidate proxy rate: {:.3f}".format(candidate_proxy_labels.mean()))
print("Top-10 proxy precision: {:.3f}".format(
    precision_at_k(candidate_proxy_labels, 10)
))
print("Top-20 proxy precision: {:.3f}".format(
    precision_at_k(candidate_proxy_labels, 20)
))
print("Interpretation: top-10 proxy precision is below the dataset base rate.")
print("This rule is a review-prioritization baseline, not a decline predictor.")


Rule: impressions >= 300 and 10 < avg_position <= 50
Score: impressions_90d for eligible pages
Reason code: visible_striking_distance
Action: review_title_content_and_search_snippet

Total dataset rows: 30000
Qualifying candidates: 10036
Candidate share: 33.45%
CSV written: C:/Users/chiru/Documents/flyrank-ml-internship/work/outputs/baseline_action_score.csv

EVALUATION ? CURRENT-SNAPSHOT DECLINE PROXY ONLY
Dataset proxy base rate: 0.542
Candidate proxy rate: 0.607
Top-10 proxy precision: 0.400
Top-20 proxy precision: 0.400
Interpretation: top-10 proxy precision is below the dataset base rate.
This rule is a review-prioritization baseline, not a decline predictor.


## 3. Top-20 review

The notebook prints the top 20 ranked rows. For each row it gives the action, why the page entered the queue, a confidence note, and what could make the recommendation wrong.

These are desk-review hypotheses based on the available snapshot, not conclusions about page quality. The assignment's required top 10 is included within this top-20 review.


In [3]:
review = candidates.head(20).copy()

print("TOP-20 HUMAN REVIEW")
for _, row in review.iterrows():
    print(
        f"\nRank {int(row['rank'])}: {row['content_id']} | "
        f"Action: {row['action']}"
    )
    print(
        "Why it is here: "
        f"{int(row['impressions_90d']):,} impressions and average "
        f"position {row['avg_position']:.1f}; it meets the visible "
        "striking-distance rule."
    )
    print(
        "Confidence: Moderate that the page has measurable search "
        "visibility; low that this snapshot alone identifies the best edit."
    )
    print(
        "What would make it wrong: the queries may be irrelevant, the "
        "snippet may already match intent, the page may have seasonal demand, "
        "or the position/CTR may vary substantially by query."
    )


TOP-20 HUMAN REVIEW

Rank 1: content_2cb567c3c89b | Action: review_title_content_and_search_snippet
Why it is here: 497,727 impressions and average position 22.2; it meets the visible striking-distance rule.
Confidence: Moderate that the page has measurable search visibility; low that this snapshot alone identifies the best edit.
What would make it wrong: the queries may be irrelevant, the snippet may already match intent, the page may have seasonal demand, or the position/CTR may vary substantially by query.

Rank 2: content_2dba2b1f9536 | Action: review_title_content_and_search_snippet
Why it is here: 443,434 impressions and average position 27.9; it meets the visible striking-distance rule.
Confidence: Moderate that the page has measurable search visibility; low that this snapshot alone identifies the best edit.
What would make it wrong: the queries may be irrelevant, the snippet may already match intent, the page may have seasonal demand, or the position/CTR may vary substantially 

## 4. Weak picks + leakage check

A rule that ranks pages by impressions can put pages in the queue even when their CTR is already comparatively strong. Those are useful weak picks to inspect: they may need a content or relevance review, but a CTR-focused fix may be unnecessary.

The decline proxy is used only after ranking to describe evaluation results. It is not a scoring input. The dataset is a single snapshot, so this proxy must not be described as a verified future decline.


In [4]:
# Flag potentially weak picks among the top 20:
# relatively high CTR can mean a CTR-focused intervention is not warranted.
top20 = candidates.head(20).copy()
weak_picks = top20[top20["ctr"] >= 0.20]

print("POTENTIAL WEAK PICKS (top 20 with CTR >= 0.20%)")
if weak_picks.empty:
    print("No rows met this review heuristic; inspect the lowest-confidence rows manually.")
else:
    for _, row in weak_picks.iterrows():
        print(
            f"Rank {int(row['rank'])}: {row['content_id']} ? "
            f"CTR {row['ctr']:.2f}%, position {row['avg_position']:.1f}. "
            "A CTR-focused fix may not be needed; inspect query relevance "
            "and page intent before recommending a change."
        )

# Explicit feature audit: only these columns are used to form the score.
scoring_inputs = [
    "impressions_90d",
    "avg_position"
]
forbidden_inputs = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]

assert not set(scoring_inputs).intersection(forbidden_inputs)
assert all(column in df.columns for column in scoring_inputs)
assert "trend_direction" not in scoring_inputs
assert "trend_pct" not in scoring_inputs
assert "content_id" not in scoring_inputs
assert "client_id" not in scoring_inputs

print("\nLEAKAGE CHECK")
print("Scoring inputs:", scoring_inputs)
print("Forbidden fields excluded from scoring:", forbidden_inputs)
print("PASS ? IDs and trend/label-derived fields are not scoring inputs.")
print("Evaluation-only field: trend_direction (current-snapshot proxy).")
print("No future-window feature is used by this rule.")


POTENTIAL WEAK PICKS (top 20 with CTR >= 0.20%)
Rank 2: content_2dba2b1f9536 ? CTR 0.21%, position 27.9. A CTR-focused fix may not be needed; inspect query relevance and page intent before recommending a change.
Rank 8: content_c5063073d048 ? CTR 0.24%, position 12.5. A CTR-focused fix may not be needed; inspect query relevance and page intent before recommending a change.
Rank 12: content_eb366e871254 ? CTR 0.20%, position 16.6. A CTR-focused fix may not be needed; inspect query relevance and page intent before recommending a change.
Rank 14: content_6a5b8ccbd700 ? CTR 0.60%, position 18.3. A CTR-focused fix may not be needed; inspect query relevance and page intent before recommending a change.
Rank 17: content_758db544d84f ? CTR 0.41%, position 13.8. A CTR-focused fix may not be needed; inspect query relevance and page intent before recommending a change.

LEAKAGE CHECK
Scoring inputs: ['impressions_90d', 'avg_position']
Forbidden fields excluded from scoring: ['trend_direction', 't

## 5. Self-check

- [x] Two signal checks have visible bucket tables and sample counts (`n`).
- [x] At least one signal is linked to a real FlyRank flag family: staleness / refresh.
- [x] Each signal has a one-word verdict with an explanation.
- [x] One transparent rule, one reason code, and one action label.
- [x] Ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] Top-20 review includes action, reason, confidence, and what could make it wrong.
- [x] Weak picks are explicitly inspected.
- [x] Trend-derived fields and IDs are excluded from scoring.
- [x] CSV is generated by the notebook and is not intended for Git.

### Honest result

The rule produces a review queue, but its top-10 current-snapshot decline-proxy precision is 0.400, below the dataset base rate of 0.542. The baseline therefore does not demonstrate useful decline ranking. Its value is as a transparent, reproducible human-review queue and a frozen comparison point for later model work.

The starter dataset does not establish a future outcome. Do not claim predictive performance or causal impact from these results.
